<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_2_ROC_AUC_Threshold_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROC Curves, AUC & Threshold Tuning

**Author:** Brad Sheese

---

## Introduction

ROC (Receiver Operating Characteristic) curves let us visualize performance across ALL thresholds (not just 0.5).

AUC (Area Under the Curve) summarizes this into a single score.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Plot and interpret ROC curves
2. Calculate and interpret AUC scores
3. Use threshold tuning to optimize for precision vs recall trade-off
4. Apply these skills to real-world decision making

## Load Data and Train Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame
y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Probability of default
y_pred = model.predict(X_test_scaled)

print("Model ready for ROC/AUC analysis!")

## Understanding ROC Curves

The ROC curve plots:
- **True Positive Rate (TPR/Recall)**: TP / (TP + FN)
- **False Positive Rate (FPR)**: FP / (FP + TN)

As we lower the threshold:
- We catch more positives (TPR increases)
- But we also get more false alarms (FPR increases)

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

print(f"AUC Score: {roc_auc:.3f}")

# Plot ROC curve
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
         label='Random Classifier (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.2, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12)
plt.title('ROC Curve: Credit Default Prediction', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nA perfect model would hug the top-left corner (AUC=1.0)")
print(f"Our model (AUC={roc_auc:.3f}) performs better than random guessing")

## AUC Interpretation

| AUC Score | Interpretation |
|-----------|----------------|
| 0.9 - 1.0 | Excellent (rare) |
| 0.8 - 0.9 | Very Good |
| 0.7 - 0.8 | Good |
| 0.6 - 0.7 | Fair |
| 0.5 - 0.6 | Poor (no better than random) |

## Precision-Recall Curve

PR curves are often better than ROC for imbalanced datasets. They focus on the positive class.

Useful when positives are rare (like fraud detection).

In [ ]:
# Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = np.trapezoid(precision, recall)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='blue', lw=2, 
         label=f'Precision-Recall Curve (AUC = {abs(pr_auc):.3f})')
plt.axhline(y=y_test.mean(), color='red', linestyle='--', 
           label=f'Baseline (always predict majority): {y_test.mean():.2f}')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve: Credit Default Prediction', fontsize=14)
plt.legend(loc='lower left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: PR curve shows the trade-off more honestly for imbalanced data")

## Threshold Tuning: Finding the Optimal Point

The optimal threshold depends on the business problem:
- High recall needed? Lower threshold
- High precision needed? Raise threshold
- Balanced? Check F1 at each threshold

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Test different thresholds
thresholds_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []
for thresh in thresholds_to_test:
    y_pred_t = (y_proba >= thresh).astype(int)
    acc = (y_pred_t == y_test).mean()
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    results.append({'threshold': thresh, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1})

results_df = pd.DataFrame(results)
print("Performance at Different Thresholds:")
print(results_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
plt.plot(results_df['threshold'], results_df['precision'], 'o-', label='Precision', linewidth=2)
plt.plot(results_df['threshold'], results_df['recall'], 's-', label='Recall', linewidth=2)
plt.plot(results_df['threshold'], results_df['f1'], '^-', label='F1-Score', linewidth=2)
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision, Recall, F1 vs. Threshold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBest F1-Score occurs at threshold = {results_df.loc[results_df['f1'].idxmax(), 'threshold']}")

## Business Example: Choosing the Right Threshold

**Scenario**: Bank deciding credit limits
- Too many defaults (low recall) = financial losses
- Too many false rejections (low precision) = lost customers

The 'optimal' threshold depends on business costs of each error type!

In [ ]:
# Visualize threshold selection by business context
# Assume: Cost of missing a default is 10x higher than rejecting a good customer
cost_ratio_fn_fp = 10  # False negative (missed default) costs 10x more

# Calculate the 'cost' at each threshold
costs = []
for thresh in thresholds_to_test:
    y_pred_t = (y_proba >= thresh).astype(int)
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()  # False alarms
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()  # Missed defaults
    total_cost = (fp + fn * cost_ratio_fn_fp) / len(y_test)
    costs.append({'threshold': thresh, 'cost': total_cost})

costs_df = pd.DataFrame(costs)

plt.figure(figsize=(10, 5))
plt.plot(costs_df['threshold'], costs_df['cost'], 'o-', color='red', linewidth=2, markersize=8)
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Normalized Cost', fontsize=12)
plt.title('Business Cost at Different Thresholds (FN costs 10x FP)', fontsize=14)
plt.grid(True, alpha=0.3)

best_thresh = costs_df.loc[costs_df['cost'].idxmin(), 'threshold']
plt.axvline(x=best_thresh, color='green', linestyle='--', label=f'Optimal threshold: {best_thresh}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Given our cost assumptions, optimal threshold = {best_thresh}")
print("In practice, you need domain expertise to estimate true costs.")

## Key Takeaways

1. **ROC Curve** visualizes TPR vs FPR at all thresholds
2. **AUC** summarizes model discrimination (1.0 = perfect, 0.5 = random)
3. **Precision-Recall Curve** better for imbalanced data
4. **Threshold tuning** lets you optimize for business goals
5. The optimal threshold depends on the cost of false positives vs. false negatives